In [95]:
from pyspark.sql.functions import broadcast, split, lit
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Homework") \
    .getOrCreate()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

spark

25/08/07 06:15:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [96]:
match_details = spark.read.option("header", "true").option("delimiter", ",").csv("/home/iceberg/data/match_details.csv")
matches = spark.read.option("header", "true").option("delimiter", ",").csv("/home/iceberg/data/matches.csv")
medals_matches_players = spark.read.option("header", "true").option("delimiter", ",").csv("/home/iceberg/data/medals_matches_players.csv")
medals = spark.read.option("header", "true").option("delimiter", ",").csv("/home/iceberg/data/medals.csv")
maps = spark.read.option("header", "true").option("delimiter", ",").csv("/home/iceberg/data/maps.csv")

In [97]:
broadcast_medals = medals_matches_players.join(broadcast(medals), on="medal_id", how="left")
broadcast_maps = matches.join(broadcast(maps), on="mapid", how="left")

In [100]:
match_details_bucketed = match_details.write.bucketBy(16, "match_id") \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("homework.match_details")

matches_bucketed = matches.write.bucketBy(16, "match_id")\
    .format("parquet")\
    .mode("overwrite")\
    .saveAsTable("homework.matches")

medals_matches_players_bucketed = medals_matches_players.write.bucketBy(16, "match_id")\
    .format("parquet")\
    .mode("overwrite")\
    .saveAsTable("homework.medals_matches_players")

match_details_bucketed = spark.read.format("parquet") \
    .load("homework.match_details")
matches_bucketed = spark.read.format("parquet") \
    .load("homework.matches")
medals_matches_players_bucketed = spark.read.format("parquet") \
    .load("homework.medals_matches_players")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/iceberg/notebooks/notebooks/bootcamp.match_details.

In [70]:
bucket_joined = match_details_bucketed.join(matches_bucketed, on="match_id") \
                                      .join(medals_matches_players_bucketed, on="match_id")

bucket_joined.explain()

In [71]:
joined = spark.sql("""
    SELECT 
        md.match_id,
        md.player_gamertag,
        md.previous_spartan_rank,
        md.spartan_rank,
        md.previous_total_xp,
        md.total_xp,
        md.previous_csr_tier,
        md.previous_csr_designation,
        md.previous_csr,
        md.previous_csr_percent_to_next_tier,
        md.previous_csr_rank,
        md.current_csr_tier,
        md.current_csr_designation,
        md.current_csr,
        md.current_csr_percent_to_next_tier,
        md.current_csr_rank,
        md.player_rank_on_team,
        md.player_finished,
        md.player_average_life,
        md.player_total_kills,
        md.player_total_headshots,
        md.player_total_weapon_damage,
        md.player_total_shots_landed,
        md.player_total_melee_kills,
        md.player_total_melee_damage,
        md.player_total_assassinations,
        md.player_total_ground_pound_kills,
        md.player_total_shoulder_bash_kills,
        md.player_total_grenade_damage,
        md.player_total_power_weapon_damage,
        md.player_total_power_weapon_grabs,
        md.player_total_deaths,
        md.player_total_assists,
        md.player_total_grenade_kills,
        md.did_win,
        md.team_id,
        
        m.mapid,
        m.is_team_game,
        m.playlist_id,
        m.game_variant_id,
        m.is_match_over,
        m.completion_date,
        m.match_duration,
        m.game_mode,
        m.map_variant_id,
        
        mmp.medal_id,
        mmp.count
    FROM match_details md
    JOIN matches m
    ON md.match_id = m.match_id
    JOIN medals_matches_players mmp
    ON m.match_id = mmp.match_id
""")

joined.write.mode("overwrite").saveAsTable("homework.bucket_join")

In [45]:
#Average most kills
avg_most_kills = joined.groupBy("player_gamertag")\
                        .agg(F.avg("player_total_kills").alias("avg_kills"))\
                        .orderBy(F.desc("avg_kills"))

avg_most_kills.show(1)

+---------------+---------+
|player_gamertag|avg_kills|
+---------------+---------+
|   gimpinator14|    109.0|
+---------------+---------+
only showing top 1 row



In [102]:
#Most played playlist

most_played_map = joined.filter(F.col("map_variant_id").isNotNull()) \
                        .groupBy("map_variant_id") \
                        .agg(F.count("*").alias("map_played_count")) \
                        .orderBy(F.desc("map_played_count"))

most_played_map.show(1)             

+--------------------+----------------+
|      map_variant_id|map_played_count|
+--------------------+----------------+
|a72c6f2e-9972-4a2...|           81008|
+--------------------+----------------+
only showing top 1 row



In [101]:
#Most played map
most_played_map = joined.groupBy("map_variant_id")\
                        .agg(F.count("*").alias("map_played_count"))\
                        .orderBy(F.desc("map_played_count"))

most_played_map.show(1)

+--------------+----------------+
|map_variant_id|map_played_count|
+--------------+----------------+
|          NULL|         3637182|
+--------------+----------------+
only showing top 1 row



In [92]:
#Map with most medals
killing_spree_id = medals.filter(F.col("name") == "Killing Spree").select("medal_id").first()["medal_id"]

highest_medals_map = joined.filter(F.col("medal_id") == killing_spree_id)\
                            .groupBy("mapid")\
                            .agg(F.count("medal_id").alias("medals_map_count"))\
                            .orderBy(F.desc("medals_map_count"))

highest_medals_map.show(1)

25/08/07 06:03:40 WARN DataSourceV2Strategy: Can't translate true to source filter, unsupported expression


+--------------------+----------------+
|               mapid|medals_map_count|
+--------------------+----------------+
|c74c9d0f-f206-11e...|           56908|
+--------------------+----------------+
only showing top 1 row



In [ ]:
sorted_by_playlist = joined.sortWithinPartitions("playlist_id")

sorted_by_map = joined.sortWithinPartitions("mapid")

print("Size sorted by playlist_id:", sorted_by_playlist.rdd.map(lambda x: len(str(x))).sum())
print("Size sorted by mapid:", sorted_by_map.rdd.map(lambda x: len(str(x))).sum())

[Stage 440:>                                                      (0 + 12) / 13]

[5675.246s][warning][gc,alloc] stdout writer for python3: Retried waiting for GCLocker too often allocating 131074 words
[5675.246s][warning][gc,alloc] stdout writer for python3: Retried waiting for GCLocker too often allocating 131074 words
[5675.248s][warning][gc,alloc] stdout writer for python3: Retried waiting for GCLocker too often allocating 131074 words
[5675.249s][warning][gc,alloc] stdout writer for python3: Retried waiting for GCLocker too often allocating 131074 words


25/08/07 06:19:43 ERROR Utils: Uncaught exception in thread stdout writer for python3
java.lang.OutOfMemoryError: Java heap space
25/08/07 06:19:43 ERROR Utils: Uncaught exception in thread stdout writer for python3
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:64)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:363)
	at org.apache.spark.io.ReadAheadInputStream.<init>(ReadAheadInputStream.java:105)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillReader.<init>(UnsafeSorterSpillReader.java:77)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillWriter.getReader(UnsafeSorterSpillWriter.java:159)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.getSortedIterator(UnsafeExternalSorter.java:555)
	at org.apache.spark.sql.execution.UnsafeExternalRowSorter.sort(UnsafeExternalRowSorter.java:172)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIterato